# TP5-6 2026 Opérateurs de jointure


date de modification: 26/02/2026

# Préparation

installer (ou mettre à jour) DuckDB


In [5]:
# !pip install duckdb --pre

# option -U pour installer la dernière version de duckDB si elle n'est pas déjà installée
!pip install -U duckdb

pour afficher les plans d'exécution sous la forme d'arbres

In [6]:
!pip install -q anytree

Les lib nécessaires pour ce TP

In [7]:
import duckdb

import sqlite3

import time
import pandas as pd

import os
import glob
import shutil as sh
import numpy as np
import random

# regexp
import re

# pretty print
from anytree import Node, RenderTree

#lecture de fichiers
# import csv

# Iterator
try:
    from collections.abc import Iterator  # For Python 3.9+
except ImportError:
    from typing import Iterator  # For Python 3.8 and below


os.makedirs("data", exist_ok=True)

connection au SGBD

In [8]:
db = duckdb.connect('tpch.db')

## Créer les données


On utilise la base de données du benchmark TPC-H.
DuckDB propose la procedure *dbgen* pour générer la base de données.
Les données sont stockées dans le fichier *tpch.db*


Comprendre le [schéma des données TPC-H](https://nuage.lip6.fr/s/RramHt5W3RomySs) (ou cf moodle).

La taille de la base est paramétrée par le scale factor *sf*, vous pouvez faire varier sa valeur de 0.05 à 1 pour créer des bases de plus en plus grandes selon les expériences que vous voulez faire.


In [48]:
# scale factor
# sf = 0.2
sf = 0.05

generate=True

# Ne pas regénérer la base si elle est existe déjà
order_table_created = db.execute("select count(*) from duckdb_tables() where table_name = 'orders'").fetchall()[0][0]
if (order_table_created == 1):
  nb_orders = db.execute("select estimated_size from duckdb_tables() where table_name = 'orders'").fetchall()[0][0]
  if nb_orders / 1500000 == sf:
    print("database already generated with sf", sf)
    generate = False

if generate:
  drop_tpch_tables = """
  DROP TABLE IF EXISTS customer;
  DROP TABLE IF EXISTS lineitem;
  DROP TABLE IF EXISTS nation;
  DROP TABLE IF EXISTS orders;
  DROP TABLE IF EXISTS part;
  DROP TABLE IF EXISTS partsupp;
  DROP TABLE IF EXISTS region;
  DROP TABLE IF EXISTS supplier;
  """
  db.execute(drop_tpch_tables)

  print("generate database with scale factor sf = ", sf)
  db.execute(f"CALL dbgen(sf={sf})")

db.execute("analyze").df()

generate database with scale factor sf =  0.05


,Success


# Consulter la BD

## Table names

Le catalogue des tables est duckdb_tables()

Il y a 8 tables pour la base TPCH.

Note: la fonction  `.df()` ajoutée à la fin de l'expression sert à importer le résultat de la requête dans un dataframe pandas afin d'avoir un rendu "ergonomique" dans colab .

In [ ]:
db.execute("select table_name, estimated_size, column_count from duckdb_tables() order by estimated_size desc").df()

,table_name,estimated_size,column_count
0,lineitem,299814,16
1,orders,75000,9
2,partsupp,40000,5
3,part,10000,9
4,customer,7500,8
5,supplier,500,7
6,nation,25,4
7,region,5,3


 La commande `SHOW TABLES` retourne le nom des tables

In [ ]:
# db.execute("SHOW TABLES").df()

La clause  `LIMIT N` est ajoutée à la fin d'une requête pour calculer seulement les N premiers tuples du résultat.

Exemple pour afficher N tuples de la table Lineitem :

In [ ]:
db.execute("SELECT * FROM lineitem LIMIT 5").df()

,l_orderkey,l_partkey,l_suppkey,l_linenumber,l_quantity,l_extendedprice,l_discount,l_tax,l_returnflag,l_linestatus,l_shipdate,l_commitdate,l_receiptdate,l_shipinstruct,l_shipmode,l_comment
0,1,7760,401,1,17.0,28351.92,0.04,0.02,N,O,1996-03-13,1996-02-12,1996-03-22,DELIVER IN PERSON,TRUCK,to beans x-ray carefull
1,1,3366,367,2,36.0,45696.96,0.09,0.06,N,O,1996-04-12,1996-02-28,1996-04-20,TAKE BACK RETURN,MAIL,according to the final foxes. qui
2,1,3185,186,3,8.0,8705.44,0.10,0.02,N,O,1996-01-29,1996-03-05,1996-01-31,TAKE BACK RETURN,REG AIR,ourts cajole above the furiou
3,1,107,233,4,28.0,28198.80,0.09,0.06,N,O,1996-04-21,1996-03-30,1996-05-16,NONE,AIR,s cajole busily above t
4,1,1202,84,5,24.0,26476.80,0.10,0.04,N,O,1996-03-30,1996-03-14,1996-04-01,NONE,FOB,"the regular, regular pa"


## Describe

La commande `DESCRIBE` retourne une description des attributs d'une table.

### Lineitem

In [70]:
db.execute("DESCRIBE Lineitem").df()

,column_name,column_type,null,key,default,extra
0,l_orderkey,BIGINT,NO,None,None,None
1,l_partkey,BIGINT,NO,None,None,None
2,l_suppkey,BIGINT,NO,None,None,None
3,l_linenumber,BIGINT,NO,None,None,None
4,l_quantity,"DECIMAL(15,2)",NO,None,None,None
5,l_extendedprice,"DECIMAL(15,2)",NO,None,None,None
6,l_discount,"DECIMAL(15,2)",NO,None,None,None
7,l_tax,"DECIMAL(15,2)",NO,None,None,None
8,l_returnflag,VARCHAR,NO,None,None,None
9,l_linestatus,VARCHAR,NO,None,None,None


### Orders

In [ ]:
db.execute("DESCRIBE Orders").df()

,column_name,column_type,null,key,default,extra
0,o_orderkey,BIGINT,NO,None,None,None
1,o_custkey,BIGINT,NO,None,None,None
2,o_orderstatus,VARCHAR,NO,None,None,None
3,o_totalprice,"DECIMAL(15,2)",NO,None,None,None
4,o_orderdate,DATE,NO,None,None,None
5,o_orderpriority,VARCHAR,NO,None,None,None
6,o_clerk,VARCHAR,NO,None,None,None
7,o_shippriority,INTEGER,NO,None,None,None
8,o_comment,VARCHAR,NO,None,None,None


### Customer

In [ ]:
db.execute("DESCRIBE Customer").df()

,column_name,column_type,null,key,default,extra
0,c_custkey,BIGINT,NO,None,None,None
1,c_name,VARCHAR,NO,None,None,None
2,c_address,VARCHAR,NO,None,None,None
3,c_nationkey,INTEGER,NO,None,None,None
4,c_phone,VARCHAR,NO,None,None,None
5,c_acctbal,"DECIMAL(15,2)",NO,None,None,None
6,c_mktsegment,VARCHAR,NO,None,None,None
7,c_comment,VARCHAR,NO,None,None,None


La commande `SUMMARIZE` donne un aperçu statistique des données d'une table.
Elle retourne pour chaque attribut, les bornes min et max du domaine, le nombre approximatif de valeurs uniques ainsi que la distribution décrite par la moyenne, l'écart type et les quartiles 25, 50 (médiane) et 75.

### Supplier


In [ ]:
db.execute("DESCRIBE Supplier").df()

,column_name,column_type,null,key,default,extra
0,s_suppkey,BIGINT,NO,None,None,None
1,s_name,VARCHAR,NO,None,None,None
2,s_address,VARCHAR,NO,None,None,None
3,s_nationkey,INTEGER,NO,None,None,None
4,s_phone,VARCHAR,NO,None,None,None
5,s_acctbal,"DECIMAL(15,2)",NO,None,None,None
6,s_comment,VARCHAR,NO,None,None,None


## Table summaries

### Lineitem summary

In [ ]:
db.execute("SUMMARIZE lineitem").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,l_orderkey,BIGINT,1,300000,78087,149623.65734755548,86554.62130435975,74848,150895,223768,299814,0.0
1,l_partkey,BIGINT,1,10000,8565,5003.511817326743,2882.4722623260864,2512,5010,7494,299814,0.0
2,l_suppkey,BIGINT,1,500,520,250.96455802597612,144.36031863583764,126,251,376,299814,0.0
3,l_linenumber,BIGINT,1,7,7,2.9989193299845907,1.7317394417004388,2,3,4,299814,0.0
4,l_quantity,"DECIMAL(15,2)",1.00,50.00,57,25.528480991548093,14.41889274948568,13,26,38,299814,0.0
5,l_extendedprice,"DECIMAL(15,2)",902.00,95399.50,89958,35843.15635343913,21924.61768108415,17551,34324,51508,299814,0.0
6,l_discount,"DECIMAL(15,2)",0.00,0.10,12,0.05006233864996298,0.03159721679299363,0,0,0,299814,0.0
7,l_tax,"DECIMAL(15,2)",0.00,0.08,10,0.040053466482552515,0.025793025564006776,0,0,0,299814,0.0
8,l_returnflag,VARCHAR,A,R,3,None,None,None,None,None,299814,0.0
9,l_linestatus,VARCHAR,F,O,2,None,None,None,None,None,299814,0.0


### Orders summary

In [ ]:
db.execute("SUMMARIZE Orders").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,o_orderkey,BIGINT,1,300000,78087,149991.5,86603.1177570997,74992,149991,224991,75000,0.0
1,o_custkey,BIGINT,1,7499,5210,3747.8847866666665,2168.4548865095485,1864,3741,5632,75000,0.0
2,o_orderstatus,VARCHAR,F,P,2,None,None,None,None,None,75000,0.0
3,o_totalprice,"DECIMAL(15,2)",821.70,478519.25,92643,141575.3585368,83126.13882759152,72716,135228,201432,75000,0.0
4,o_orderdate,DATE,1992-01-01,1998-08-02,2321,1995-04-19 01:11:12.768,None,1993-08-29,1995-04-19,1996-12-09,75000,0.0
5,o_orderpriority,VARCHAR,1-URGENT,5-LOW,5,None,None,None,None,None,75000,0.0
6,o_clerk,VARCHAR,Clerk#000000001,Clerk#000001000,860,None,None,None,None,None,75000,0.0
7,o_shippriority,INTEGER,0,0,1,0.0,0.0,0,0,0,75000,0.0
8,o_comment,VARCHAR,Tiresias sleep stealthily unusual theodolites...,zzle: slyly even ideas wake furiously across t...,94261,None,None,None,None,None,75000,0.0


### Customer summary

In [ ]:
db.execute("SUMMARIZE Customer").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,c_custkey,BIGINT,1,7500,6841,3750.5,2165.2078422174627,1876,3750,5626,7500,0.0
1,c_name,VARCHAR,Customer#000000001,Customer#000007500,6007,None,None,None,None,None,7500,0.0
2,c_address,VARCHAR,3SUEwTaEs35ZDr5dx6P2b,"zzB4DRh4Eg3uFygL7UZZMiBaR,Lbep8 IOreIH2",7967,None,None,None,None,None,7500,0.0
3,c_nationkey,INTEGER,0,24,26,11.913066666666667,7.232146969528544,6,12,18,7500,0.0
4,c_phone,VARCHAR,10-103-318-6809,34-999-121-7718,9129,None,None,None,None,None,7500,0.0
5,c_acctbal,"DECIMAL(15,2)",-999.95,9999.72,7262,4443.791336,3179.4579298123053,1736,4353,7232,7500,0.0
6,c_mktsegment,VARCHAR,AUTOMOBILE,MACHINERY,4,None,None,None,None,None,7500,0.0
7,c_comment,VARCHAR,"Tiresias haggle furiously bold, express instr...",zzle furiously regular foxes:,7738,None,None,None,None,None,7500,0.0


### Supplier summary

In [ ]:
db.execute("SUMMARIZE Supplier").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,s_suppkey,BIGINT,1,500,520,250.5,144.4818327679989,126,250,376,500,0.0
1,s_name,VARCHAR,Supplier#000000001,Supplier#000000500,488,None,None,None,None,None,500,0.0
2,s_address,VARCHAR,0W7IPdkpWycUbQ9Adp6B,zaux5FTzToEg,540,None,None,None,None,None,500,0.0
3,s_nationkey,INTEGER,0,24,26,12.33,7.142768250234156,7,12,18,500,0.0
4,s_phone,VARCHAR,10-119-372-1062,34-885-883-5717,561,None,None,None,None,None,500,0.0
5,s_acctbal,"DECIMAL(15,2)",-966.20,9957.00,537,4394.91834,3149.596777416193,1663,4412,6956,500,0.0
6,s_comment,VARCHAR,accounts cajole quickly bold foxes. slyly spe...,"yly even accounts nag ironic, even pinto beans...",468,None,None,None,None,None,500,0.0


# Plan d'une requête : explain_query
On peut visualiser le **plan** d'exécution d'une requête en préfixant la requête par `EXPLAIN`.

La commande `EXPLAIN` ne fait qu'afficher le plan d'exécution ; elle n'exécute **pas** la requête.

Le plan affiché est appelé *plan physique* car il indique pour chaque opérateur du plan, le nom de l'algorithme utilisé:
*   HASH_GROUP_BY : regroupement par hachage puis agrégation
*   HASH_JOIN : jointure par hachage
*   FILTER : sélection en pipeline
*   SEQ_SCAN : lecture séquentielle d'une table

De plus le plan indique la cardinalité estimée **~n Rows** pour certains opérateurs, en particulier pour les sélections et les jointures.

In [49]:
def explain_query(query):
  print(db.execute("EXPLAIN " + query).fetchall()[0][1])

print("fonction explain_query définie")

fonction explain_query définie


Remarque : vous pourriez invoquer `PRAGMA explain_output='all'` pour afficher, en plus du plan physique, le plan logique avant son optimisation et le plan logique optimisé, mais ce n'est pas demandé dans ce TP.

plan d'une requête

In [50]:
query = """
SELECT o_orderkey, o_totalprice, c_name
FROM customer JOIN orders ON (c_custkey=o_custkey)
WHERE c_mktsegment = 'BUILDING'
  AND o_totalprice < 1000
"""

print(query)
explain_query(query)


SELECT o_orderkey, o_totalprice, c_name
FROM customer JOIN orders ON (c_custkey=o_custkey)
WHERE c_mktsegment = 'BUILDING'
  AND o_totalprice < 1000

┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│         o_orderkey        │
│        o_totalprice       │
│           c_name          │
│                           │
│        ~3,750 rows        │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         HASH_JOIN         │
│    ────────────────────   │
│      Join Type: INNER     │
│                           │
│        Conditions:        ├──────────────┐
│   o_custkey = c_custkey   │              │
│                           │              │
│        ~3,750 rows        │              │
└─────────────┬─────────────┘              │
┌─────────────┴─────────────┐┌─────────────┴─────────────┐
│          SEQ_SCAN         ││           FILTER          │
│    ────────────────────   ││    ────────────────────   │
│           Table:         

# Exécuter une requête: run_query

On définit la fonction `run_query` pour **exécuter** une requête et mesurer sa durée en millisecondes (durée moyenne sur n exécutions)

In [51]:
# par défaut on exécute la requête dans DuckDB (db).
# On execute la requête 3 fois pour avoir une durée moyenne
def run_query(q, connection=db, iteration=3):
  start = time.time()
  n = 0
  for i in range(iteration):
    res = connection.execute(q).fetchall()
    # res = connection.execute(q).arrow()
    # res = connection.execute(q).df()
    n = len(res)

  end = time.time()
  print(f"{n} tuples(s) produced in {(end - start)*1000/iteration:.0f}ms (avg for {iteration} iter.)")
  print(res)

print("fonction run_query définie")

fonction run_query définie


Exécuter une requête

In [52]:
query = """
SELECT o_orderkey, o_totalprice, c_name
FROM customer JOIN orders ON (c_custkey=o_custkey)
WHERE c_mktsegment = 'BUILDING'
  AND o_totalprice < 1000
"""

# query = """
# SELECT
#     l_linenumber, l_extendedprice, o_orderkey, o_totalprice, c_name
# FROM
#     customer
#     JOIN orders ON (c_custkey=o_custkey)
#     JOIN lineitem ON (l_orderkey=o_orderkey)
# WHERE
#     c_mktsegment = 'BUILDING'
# AND o_totalprice < 1000
# """

run_query(query)

5 tuples(s) produced in 5ms (avg for 3 iter.)
[(174433, Decimal('954.92'), 'Customer#000007069'), (241991, Decimal('991.06'), 'Customer#000000580'), (244673, Decimal('994.31'), 'Customer#000005152'), (263364, Decimal('962.16'), 'Customer#000005585'), (267430, Decimal('989.49'), 'Customer#000001550')]


# Rapport après exécution

On peut afficher un rapport **après** l'exécution d'un plan pour connaitre le nombre de nuplets (*Rows*) produits par chaque opération du plan. On appelle cela le *profil* de l'exécution.

La fonction  `run_and_profile_query` active le mode profil afin d'enregistrer les informations de profil.
La requête est ensuite exécutée puis les informations de profil enregistrées pendant l'exécution sont affichées.

On peut lire dans le profil, pour chaque opérateur, la durée écoulée, le nombre de tuples consommés en entrée ainsi que le nombre de tuples produits pour l'opérateur suivant.

Plus spécifiquement, l'opérateur de lecture séquentielle `SEQ_SCAN` (sequential scan) indique le nombre de tuples lus dans une table.
L'opérateur `HASH_GROUP_BY` indique le nombre de groupes créés.

Remarque: pour les opérateurs qui matérialisent leur sortie, le nombre de tuples produits représente la cardinalité du résultat intermédiaire.

Les cardinalités intermédiaires sont importantes car elles peuvent expliquer pourquoi un opérateur a pris beaucoup de temps pour être traité.
Dans de nombreuses situations, il est possible de réduire la cardinalité des résultats intermédiaires et modifiant l'ordre des opérations et la façon dont elles sont traitées.


In [53]:
def run_and_profile_query(query):
  db.execute("PRAGMA enable_profiling")
  db.execute("PRAGMA profiling_output='out.txt'")
  # res = db.execute(query).arrow()
  # res = db.execute(query).df()
  res = db.execute(query).fetchall()
  db.execute("PRAGMA disable_profiling")
  print(len(res), "tuple(s)")
  with open('out.txt', 'r') as f:
    output = f.read()
  print(output)

run_and_profile_query(query)

5 tuple(s)
┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
 SELECT o_orderkey, o_totalprice, c_name FROM customer JOIN orders ON (c_custkey=o_custkey) WHERE c_mktsegment = 'BUILDING'   AND o_totalprice < 1000 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.0064s             ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│         o_orderkey        │
│        o_totalprice       │
│           c_name          │
│                           │
│                           │
│                           │
│           5 rows     

# Exercice 1 : Plan d'exécution dans DuckDB



On donne l'expression algébrique d'une requête de jointure entre Lineitem et Orders avec des sélections pour réduire la taille des relations.
On a :
*   $R1 = T_1 \bowtie T_2$ avec :
*   $T_1 = \pi_{l\_linenumber, l\_orderkey} (\sigma_{p_1}(Lineitem))$
*   $T_2 = \pi_{o\_orderkey, o\_custkey} (\sigma_{p_2}(Orders))$

et avec les prédicats de sélection :
*   $p_1: l\_extendedprice \le 902$
*   $p_2: o\_totalprice \ge 1000$





## Question 1: Ecrire R1 en SQL et mesurer sa durée

In [ ]:
query= """
SELECT l_linenumber, l_orderkey, o_custkey
FROM Lineitem
JOIN Orders  on o_orderkey = l_orderkey
WHERE
l_extendedprice <= 902
AND
o_totalprice >= 1000

"""

db.execute("PRAGMA enable_optimizer")

explain_query(query)
run_query(query)

┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│        l_linenumber       │
│         l_orderkey        │
│         o_custkey         │
│                           │
│        ~11,992 rows       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         HASH_JOIN         │
│    ────────────────────   │
│      Join Type: INNER     │
│                           │
│        Conditions:        ├──────────────┐
│  l_orderkey = o_orderkey  │              │
│                           │              │
│        ~11,992 rows       │              │
└─────────────┬─────────────┘              │
┌─────────────┴─────────────┐┌─────────────┴─────────────┐
│          SEQ_SCAN         ││          SEQ_SCAN         │
│    ────────────────────   ││    ────────────────────   │
│           Table:          ││           Table:          │
│     tpch.main.lineitem    ││      tpch.main.orders     │
│                           ││                           │
│

## Question 2. Quelles sont les cardinalités estimées et réelles des opérandes  $T_1$ et $T_2$ ?

cardinalité estimée et réelle de T1

In [ ]:
T1 = """
select l_linenumber, l_orderkey
from Lineitem
where l_extendedprice <= 902
"""

db.execute("PRAGMA enable_optimizer")
print("estimation:")
explain_query(T1)
print("réel:")
run_query(T1)

estimation:
┌───────────────────────────┐
│          SEQ_SCAN         │
│    ────────────────────   │
│           Table:          │
│     tpch.main.lineitem    │
│                           │
│   Type: Sequential Scan   │
│                           │
│        Projections:       │
│        l_linenumber       │
│         l_orderkey        │
│                           │
│          Filters:         │
│  l_extendedprice<=902.00  │
│                           │
│        ~59,962 rows       │
└───────────────────────────┘

réel:
2 tuples(s) produced in 1ms (avg for 3 iter.)


cardinalité estimée et réelle de T2

In [ ]:
T2 = """
select o_orderkey, o_custkey
from Orders
where o_totalprice >= 1000
"""

print("estimation:")
explain_query(T2)
print("réel:")
run_query(T2)


estimation:
┌───────────────────────────┐
│          SEQ_SCAN         │
│    ────────────────────   │
│           Table:          │
│      tpch.main.orders     │
│                           │
│   Type: Sequential Scan   │
│                           │
│        Projections:       │
│         o_orderkey        │
│         o_custkey         │
│                           │
│          Filters:         │
│   o_totalprice>=1000.00   │
│                           │
│        ~15,000 rows       │
└───────────────────────────┘

réel:
74976 tuples(s) produced in 21ms (avg for 3 iter.)


## Question 3 : Modifier l'ordre des relations dans une jointure

On désactive l'optimiseur de DuckDB. Ainsi, le plan conserve l'ordre des relations qui est spécifié dans la requête SQL.

Soit la requête:

`SELECT * FROM Relation1 JOIN Relation2 ON (...)`

alors dans le plan, la `Relation1` sera placée à gauche et la `Relation2` sera placée à droite.

Indication: Relation1 et Relation2 doivent être exprimées comme des **sous-requêtes** imbriquées dans la requpête principale



### a) Lire la plus petite relation en premier

Réécrire R en une requête SQL équivalente pour que :
*    les sélections soient traitées avant la jointure.
*    La relation la plus petite $T_1$ (car $T_1$ << $T_2$) soit chargée en mémoire avant de lire $T_2$. Donc $T_1$ doit donc être l'opérande de **droite** de l'opérateur hash join d'après la convention suivie par DuckDB.

In [ ]:
query= """
SELECT *
FROM (
     SELECT o_orderkey, o_custkey
     FROM Orders
     WHERE o_totalprice >= 1000
    ) T2

JOIN (
    select l_linenumber, l_orderkey
    from Lineitem
    where l_extendedprice <= 902
) T1
ON  o_orderkey = l_orderkey
"""



# db.execute("PRAGMA enable_optimizer")
db.execute("PRAGMA disable_optimizer")

explain_query(query)
run_query(query)


┌───────────────────────────┐
│         HASH_JOIN         │
│    ────────────────────   │
│      Join Type: INNER     │
│                           │
│        Conditions:        ├──────────────┐
│  o_orderkey = l_orderkey  │              │
│                           │              │
│          ~0 rows          │              │
└─────────────┬─────────────┘              │
┌─────────────┴─────────────┐┌─────────────┴─────────────┐
│         PROJECTION        ││         PROJECTION        │
│    ────────────────────   ││    ────────────────────   │
│         o_orderkey        ││        l_linenumber       │
│         o_custkey         ││         l_orderkey        │
│                           ││                           │
│        ~75,000 rows       ││       ~299,814 rows       │
└─────────────┬─────────────┘└─────────────┬─────────────┘
┌─────────────┴─────────────┐┌─────────────┴─────────────┐
│           FILTER          ││           FILTER          │
│    ────────────────────   ││    ─

b) Est-ce que ce plan est plus rapide que le plan optimisé par DuckDB ?

Oui , 1.5 à 2 fois plus rapide.

# Exercice 2 : Conception de plans d'exécution

L'objectif de cet exercice est de comprendre le fonctionnement d'un plan d'exécution composé de:
* **méthodes d'accès** aux données
* **opérateurs** de sélection, projection et jointure.

Pour cela, on suppose un environnement *ouvert* en dehors d'un SGBD particulier, et vous allez construire votre propre moteur de requêtes!

Savoir faire attendus:
*   Savoir implémenter une méthode d'accès aux données avec index
*   Savoir implémenter les principaux opérateurs
*   Savoir construire un plan en assemblant des opérateurs
*   Définir plusieurs plans equivalents pour une requête et comparer leur durée



Les données sont simplement stockées dans des pages représentées par des fichiers.


## Exporter les données vers des fichiers

In [54]:
def créer_fichier(table, db=db, batch_multiple_of_2048 = 5):
  dir = os.path.join("data", table)
  if os.path.exists(dir):
    sh.rmtree(dir)
  os.makedirs(dir, exist_ok=True)


  print("batch size:", batch_multiple_of_2048 * 2048)

  query = f"SELECT * FROM {table}"
  exec = db.execute(query)

  page=0
  lines=0
  while True:
    assert(page<1000), "trop de pages pour le tp"
    df = exec.fetch_df_chunk(batch_multiple_of_2048)
    if df.empty:  # quitter si aucun tuple
        break
    page += 1
    lines += len(df)
    # print(f"batch de {len(df)} lignes")
    df.to_parquet(os.path.join("data", table, f"p{page:03}.parquet"), index=False)

  print(f"{page} pages créées dans le dossier {table} pour {lines} tuples ")

créer_fichier("lineitem", db)
créer_fichier("orders", db)
créer_fichier("customer", db)
créer_fichier("supplier", db)
créer_fichier("nation", db)

batch size: 10240
30 pages créées dans le dossier lineitem pour 299814 tuples 
batch size: 10240
8 pages créées dans le dossier orders pour 75000 tuples 
batch size: 10240
1 pages créées dans le dossier customer pour 7500 tuples 
batch size: 10240
1 pages créées dans le dossier supplier pour 500 tuples 
batch size: 10240
1 pages créées dans le dossier nation pour 25 tuples 


### Fonction utilitaire pour afficher un extrait d'un fichier

In [55]:
def affiche_debut(table, num=3):
  pages = sorted(glob.glob(os.path.join("data", table, "p*")))
  assert(len(pages)>0), "aucune page"
  p0 = pages[0]
  print(f"Les {num} premiers nuplets du fichier {p0} :")

  head = pd.read_parquet(p0).head(num)
  # entête : nom des attributs
  print("attributs:")
  print(head.columns.tolist())
  print("nuplets:")
  for r in head.to_numpy():
    print(r.tolist())
  print()

affiche_debut("orders")
affiche_debut("lineitem")
affiche_debut("customer")


Les 3 premiers nuplets du fichier data/orders/p001.parquet :
attributs:
['o_orderkey', 'o_custkey', 'o_orderstatus', 'o_totalprice', 'o_orderdate', 'o_orderpriority', 'o_clerk', 'o_shippriority', 'o_comment']
nuplets:
[1, 1846, 'O', 182897.0, Timestamp('1996-01-02 00:00:00'), '5-LOW', 'Clerk#000000951', 0, 'ly express platelets. deposits acc']
[2, 3901, 'O', 48450.57, Timestamp('1996-12-01 00:00:00'), '1-URGENT', 'Clerk#000000880', 0, 've the furiously fluffy dependencies. carefully regular ']
[3, 6166, 'F', 220604.58, Timestamp('1993-10-14 00:00:00'), '5-LOW', 'Clerk#000000955', 0, ' after the asymptotes. instructions cajole after the foxes. carefully unu']

Les 3 premiers nuplets du fichier data/lineitem/p001.parquet :
attributs:
['l_orderkey', 'l_partkey', 'l_suppkey', 'l_linenumber', 'l_quantity', 'l_extendedprice', 'l_discount', 'l_tax', 'l_returnflag', 'l_linestatus', 'l_shipdate', 'l_commitdate', 'l_receiptdate', 'l_shipinstruct', 'l_shipmode', 'l_comment']
nuplets:
[1, 7760, 40

## Question 1 : Implémenter les opérateurs de l'algèbre

Un opérateur est une sous classe de Operator.

On demande de compléter l'implémentation des opérateurs TableScan, Selection et HashJoin

In [56]:
class Operator:
  def __init__(self):
    pass

  # produire le résultat
  def iterate(self) -> Iterator[np.ndarray]:
    pass

  # le schéma relationnel du résultat produit par l'opérateur, sous la forme d'une liste d'attributs
  def attributes(self) -> list[str]:
    pass

  # afficher l'opérateur
  def __str__(self):
    pass

  # Afficher l'arbre du plan
  def pretty_print_node(self):
    pass


### class TableScan

In [24]:
class TableScan(Operator):
  def __init__(self, table):
    self.table = table
    self.pages = sorted(glob.glob(os.path.join("data", table, "p*"))) # on peut utiliser sorted,  glob.glob et os.path.join

  def iterate(self) -> Iterator[np.ndarray]:
    for page in self.pages:
      npa = pd.read_parquet(page).to_numpy()
      for row in npa:
        yield row

  def attributes(self) -> list:
    return pd.read_parquet(self.pages[0]).columns.tolist()

  def __str__(self):
    return f"TableScan({self.table})"

  def get_table(self):
    return self.table

  def pretty_print_node(self):
    return Node(f"TableScan({self.table})")

### class Selection

In [72]:
class Selection(Operator):
  def __init__(self, input: Operator, predicate):
    self.input = input
    self.predicate = predicate

    # remplace l'égalité SQL "=" par l'égalité python "=="
    predicate_in_python = re.sub(" = ", " == ", predicate) # on peut utiliser re.sub

    # remplacer le nom du ième attribut par row[i]
    attribut = predicate.split()[0]
    indice = input.attributes().index(attribut)
    predicate_in_python = re.sub(attribut, f"row[{indice}]", predicate_in_python)

    #la fonction qui évalue le prédicat
    self.check_predicate = eval(f"lambda row: {predicate_in_python}") # on peut utiliser eval

  def iterate(self) -> Iterator[np.ndarray]:
    for row in self.input.iterate():
      if self.check_predicate(row):
        yield row

  def attributes(self) -> list:
    return self.input.attributes()

  def __str__(self):
    return f"Selection({self.input}, {self.predicate})"

  def pretty_print_node(self):
    n = Node(f"Selection: {self.predicate}")
    child = self.input.pretty_print_node()
    child.parent = n
    return n

print("classe Selection définie")

classe Selection définie


### class HashJoin

In [26]:
class HashJoin(Operator):
  def __init__(self, left: Operator, right: Operator, join_predicate):
    self.left = left
    self.right = right
    self.join_predicate = join_predicate

    self.left_attribute = left.attributes()
    self.right_attribute = right.attributes()

  def iterate(self) -> Iterator[np.ndarray]:
    # Convention: matérialiser l'opérande de DROITE (meme convention que DuckDB)
    dict = {}

    attribut1 = self.join_predicate.split()[2]
    indice1 = self.right.attributes().index(attribut1)
    for right_row in self.right.iterate():
      attr = right_row[indice1]

      if attr in dict:
        dict[attr].append(right_row)
      else:
        dict[attr] = [right_row]

    attribut2 = self.join_predicate.split()[0]
    indice2 = self.left.attributes().index(attribut2)
    for left_row in self.left.iterate():
      attr = left_row[indice2]

      if attr in dict:
        for item in dict[attr]:
          yield np.concatenate((item, left_row))

  def attributes(self) -> list:
    return self.right_attribute + self.left_attribute

  def __str__(self):
    return f"HashJoin({self.left}, {self.right}, {self.join_predicate})"

  def pretty_print_node(self):
    n = Node(f"HashJoin: {self.join_predicate}")
    left = self.left.pretty_print_node()
    left.parent = n
    right = self.right.pretty_print_node()
    right.parent = n
    return n

print("classe HashJoin définie")

classe HashJoin définie


### class Limit

In [27]:
# produire seulement les n premiers tuples (cf la clause LIMIT de SQL)
class Limit(Operator):
  def __init__(self, input: Operator, nb_rows=5):
    self.input = input
    self.nb_rows = nb_rows

  def iterate(self) -> Iterator[np.ndarray]:
    for i, row in enumerate(self.input.iterate()):
      if i == self.nb_rows:
        break
      yield row

  def attributes(self) -> list:
    return self.input.attributes()

  def __str__(self):
    return f"Limit({self.input}, {self.nb_rows})"

  def pretty_print_node(self):
    n= Node(f"Limit: {self.nb_rows}")
    child = self.input.pretty_print_node()
    child.parent = n
    return n


## Fonctions utilitaires pour afficher et exécuter un plan

In [57]:
# Print tree structure
def pretty_print(plan):
  for pre, _, node in RenderTree(plan.pretty_print_node()):
    print(f"{pre}{node.name}")
  print()

def execute_plan(plan):
  for t in plan.iterate():
    print(t.tolist())


def duree_plan(plan):
  start = time.time()

  # iteration pour produire TOUS les tuples du résultat
  nbrows = sum(1 for _ in plan.iterate())

  duree = time.time() - start
  if duree >= 1:
    # durée en secondes
    print(f"{nbrows} nuplets produits en {(time.time() - start):.1f} s")
  else:
    # durée en millisecondes
    print(f"{nbrows} nuplets produits en {(time.time() - start)*1000:.0f} ms")
  print()

print("fonctions définies")

fonctions définies


## Question 2 : Construire des plans

Etant donné une requête SQL, on demande de définir un plan pour cette requête, en instanciant et en asemblant des opérateurs.

### a) Plan pour lire une table

In [ ]:
# SELECT * FROM ORDERS
plan = TableScan("orders")

execute_plan(Limit(plan,3))
duree_plan(plan)
pretty_print(plan)

[1, 1846, 'O', 182897.0, Timestamp('1996-01-02 00:00:00'), '5-LOW', 'Clerk#000000951', 0, 'ly express platelets. deposits acc']
[2, 3901, 'O', 48450.57, Timestamp('1996-12-01 00:00:00'), '1-URGENT', 'Clerk#000000880', 0, 've the furiously fluffy dependencies. carefully regular ']
[3, 6166, 'F', 220604.58, Timestamp('1993-10-14 00:00:00'), '5-LOW', 'Clerk#000000955', 0, ' after the asymptotes. instructions cajole after the foxes. carefully unu']
75000 nuplets produits en 104 ms

TableScan(orders)



In [ ]:
# SELECT * FROM LINEITEM
plan = TableScan("lineitem")

execute_plan(Limit(plan,3))
duree_plan(plan)

[1, 7760, 401, 1, 17.0, 28351.92, 0.04, 0.02, 'N', 'O', Timestamp('1996-03-13 00:00:00'), Timestamp('1996-02-12 00:00:00'), Timestamp('1996-03-22 00:00:00'), 'DELIVER IN PERSON', 'TRUCK', 'to beans x-ray carefull']
[1, 3366, 367, 2, 36.0, 45696.96, 0.09, 0.06, 'N', 'O', Timestamp('1996-04-12 00:00:00'), Timestamp('1996-02-28 00:00:00'), Timestamp('1996-04-20 00:00:00'), 'TAKE BACK RETURN', 'MAIL', ' according to the final foxes. qui']
[1, 3185, 186, 3, 8.0, 8705.44, 0.1, 0.02, 'N', 'O', Timestamp('1996-01-29 00:00:00'), Timestamp('1996-03-05 00:00:00'), Timestamp('1996-01-31 00:00:00'), 'TAKE BACK RETURN', 'REG AIR', 'ourts cajole above the furiou']
299814 nuplets produits en 799 ms



### b) Plan pour une sélection

In [29]:
# SELECT *
# FROM ORDERS
# WHERE o_totalprice >= 1000

plan = Selection(TableScan("orders"), "o_totalprice >= 1000")

execute_plan(Limit(plan,3))
duree_plan(plan)
pretty_print(plan)

[1, 1846, 'O', 182897.0, Timestamp('1996-01-02 00:00:00'), '5-LOW', 'Clerk#000000951', 0, 'ly express platelets. deposits acc']
[2, 3901, 'O', 48450.57, Timestamp('1996-12-01 00:00:00'), '1-URGENT', 'Clerk#000000880', 0, 've the furiously fluffy dependencies. carefully regular ']
[3, 6166, 'F', 220604.58, Timestamp('1993-10-14 00:00:00'), '5-LOW', 'Clerk#000000955', 0, ' after the asymptotes. instructions cajole after the foxes. carefully unu']
74976 nuplets produits en 439 ms

Selection: o_totalprice >= 1000
└── TableScan(orders)



In [ ]:
# SELECT *
# FROM LINEITEM
# WHERE l_extendedprice <= 902

plan = Selection(TableScan("lineitem"), "l_extendedprice <= 902")

execute_plan(Limit(plan,3))
duree_plan(plan)

[27939, 2000, 257, 6, 1.0, 902.0, 0.08, 0.05, 'A', 'F', Timestamp('1992-05-10 00:00:00'), Timestamp('1992-05-26 00:00:00'), Timestamp('1992-06-09 00:00:00'), 'COLLECT COD', 'TRUCK', 'f the pinto beans are about the slyl']
[131300, 2000, 1, 3, 1.0, 902.0, 0.07, 0.08, 'N', 'O', Timestamp('1995-10-20 00:00:00'), Timestamp('1995-11-18 00:00:00'), Timestamp('1995-10-22 00:00:00'), 'NONE', 'FOB', ' instructions haggle car']
2 nuplets produits en 1.3 s



### c) Plan pour une jointure

Porposer deux plans équivalentes pour cette requête

  SELECT *

  FROM lineitem JOIN orders on l_orderkey = o_orderkey

  WHERE l_extendedprice <= 902 AND o_totalprice >= 1000

In [ ]:
# SELECT *
# FROM lineitem JOIN orders on l_orderkey = o_orderkey
# WHERE l_extendedprice <= 902 AND o_totalprice >= 1000

p1 = HashJoin(Selection(TableScan("orders"), "o_totalprice >= 1000"), Selection(TableScan("lineitem"), "l_extendedprice <= 902"), "o_orderkey = l_orderkey")
pretty_print(p1)
duree_plan(p1)


p2 = HashJoin(Selection(TableScan("lineitem"), "l_extendedprice <= 902"), Selection(TableScan("orders"), "o_totalprice >= 1000"), "l_orderkey = o_orderkey")
pretty_print(p2)
duree_plan(p2)

# plan = ................................
# pretty_print(plan)
# duree_plan(plan)


HashJoin: o_orderkey = l_orderkey
├── Selection: o_totalprice >= 1000
│   └── TableScan(orders)
└── Selection: l_extendedprice <= 902
    └── TableScan(lineitem)

l_orderkey
o_orderkey
2 nuplets produits en 1.5 s

HashJoin: l_orderkey = o_orderkey
├── Selection: l_extendedprice <= 902
│   └── TableScan(lineitem)
└── Selection: o_totalprice >= 1000
    └── TableScan(orders)

o_orderkey
l_orderkey
2 nuplets produits en 1.7 s



Vérifier que le résultat est identique à celui calculé avec DuckDB

In [ ]:
run_query("""
SELECT *
FROM lineitem JOIN orders on l_orderkey = o_orderkey
WHERE l_extendedprice <= 902 AND o_totalprice >= 1000
""")

# db.execute("""
# SELECT *
# FROM lineitem JOIN orders on l_orderkey = o_orderkey
# WHERE l_extendedprice <= 902 AND o_totalprice >= 1000
# """).df()

2 tuples(s) produced in 25ms (avg for 3 iter.)
[(27939, 2000, 257, 6, Decimal('1.00'), Decimal('902.00'), Decimal('0.08'), Decimal('0.05'), 'A', 'F', datetime.date(1992, 5, 10), datetime.date(1992, 5, 26), datetime.date(1992, 6, 9), 'COLLECT COD', 'TRUCK', 'f the pinto beans are about the slyl', 27939, 5297, 'F', Decimal('167055.84'), datetime.date(1992, 4, 11), '3-MEDIUM', 'Clerk#000000977', 0, ' to the platelets. '), (131300, 2000, 1, 3, Decimal('1.00'), Decimal('902.00'), Decimal('0.07'), Decimal('0.08'), 'N', 'O', datetime.date(1995, 10, 20), datetime.date(1995, 11, 18), datetime.date(1995, 10, 22), 'NONE', 'FOB', ' instructions haggle car', 131300, 1165, 'O', Decimal('100957.91'), datetime.date(1995, 9, 5), '3-MEDIUM', 'Clerk#000000479', 0, 'y above the carefully bold excuses. furiousl')]


### d) Plan pour plusieurs jointures

Proposer des plans équivalents pour cette requête, en traitant les sélections puis les projections avant les jointures, autant que possible, en en considétant tous les ordres de jointure possibles.


Vérifier que le résultat de chaque plan est le même. Comparer leur durée.



In [ ]:
query= """
SELECT  l_orderkey, l_linenumber, s_name, s_comment, o_totalprice
FROM Lineitem
JOIN Orders on o_orderkey = l_orderkey
JOIN Supplier on l_suppkey = s_suppkey
WHERE o_totalprice < 5000
AND l_extendedprice < 1000
AND s_nationkey=13
"""
print(f"requête: {query}")
run_query(query)

print("--------")

T1 = Selection(TableScan("orders"), "o_totalprice < 5000")
T2 = Selection(TableScan("lineitem"), "l_extendedprice < 1000")
T3 = Selection(TableScan("supplier"), "s_nationkey = 13")

plan1 = HashJoin(HashJoin(T1, T2, "o_orderkey = l_orderkey"), T3, "l_suppkey = s_suppkey")
pretty_print(plan1)
execute_plan(Limit(plan1,3))
duree_plan(plan1)

print("--------------------------------------------------------------")

plan2 = HashJoin(HashJoin(T3, T2, "s_suppkey = l_suppkey"), T1, "l_orderkey = o_orderkey")
pretty_print(plan2)
execute_plan(Limit(plan2,3))
duree_plan(plan2)

requête: 
SELECT  l_orderkey, l_linenumber, s_name, s_comment, o_totalprice
FROM Lineitem
JOIN Orders on o_orderkey = l_orderkey
JOIN Supplier on l_suppkey = s_suppkey
WHERE o_totalprice < 5000
AND l_extendedprice < 1000
AND s_nationkey=13

1 tuples(s) produced in 23ms (avg for 3 iter.)
[(263364, 1, 'Supplier#000000279', 'nic deposits. blithely final requests are furiously at the ideas. carefull', Decimal('962.16'))]
--------
HashJoin: l_suppkey = s_suppkey
├── HashJoin: o_orderkey = l_orderkey
│   ├── Selection: o_totalprice < 5000
│   │   └── TableScan(orders)
│   └── Selection: l_extendedprice < 1000
│       └── TableScan(lineitem)
└── Selection: s_nationkey = 13
    └── TableScan(supplier)

[279, 'Supplier#000000279', 'R5k5c04dv7nGlvj0dJQafhoTxE,mIscBA', 13, '23-617-962-7353', 2613.77, 'nic deposits. blithely final requests are furiously at the ideas. carefull', 263364, 28, 279, 1, 1.0, 928.02, 0.04, 0.08, 'R', 'F', Timestamp('1994-08-31 00:00:00'), Timestamp('1994-09-23 00:00:00')

# Exercice 3  : Projection

Implémenter l'opérateur de projection.


### class Projection

In [81]:
class Projection(Operator):
  def __init__(self, input: Operator, predicate):
    self.input = input
    self.predicate = predicate
    self.indices = [input.attributes().index(attribut) for attribut in self.predicate.split(", ")]

  def iterate(self) -> Iterator[np.ndarray]:
    for row in self.input.iterate():
      yield [row[i] for i in self.indices]

  def attributes(self) -> list:
    result = [self.input.attributes()[i] for i in self.indices]
    return result

  def __str__(self):
    return f"Projection({self.input}, {self.predicate})"

  def pretty_print_node(self):
    n = Node(f"Projection: {self.predicate}")
    child = self.input.pretty_print_node()
    child.parent = n
    return n

print("classe Projection définie")

classe Projection définie


Construire un plan pour la requête $R_1$ (cf exercice 1) tel que les projections soient traitées le plus tôt possible.

In [83]:
query= """
SELECT l_linenumber, l_orderkey, o_custkey
FROM Lineitem
JOIN Orders  on o_orderkey = l_orderkey
WHERE
l_extendedprice <= 902
AND
o_totalprice >= 1000

"""

run_query(query)
print("--------")

# On est obligé de garder un peu plus d'attributs que ce qu'on souhaiterait car on fait des sélections et jointures.
# On pourrait refaire une deuxième projection pour s'en débarasser à la fin, si on avait une requête coûteuse utilisant ce plan.

aaa = HashJoin(Selection(Projection(TableScan("orders"), "o_orderkey, o_custkey, o_totalprice"), "o_totalprice >= 1000"), Selection(Projection(TableScan("lineitem"), "l_linenumber, l_orderkey, l_extendedprice"), "l_extendedprice <= 902"), "o_orderkey = l_orderkey")
pretty_print(aaa)
execute_plan(Limit(aaa,10))
duree_plan(aaa)

2 tuples(s) produced in 8ms (avg for 3 iter.)
[(6, 27939, 5297), (3, 131300, 1165)]
--------
HashJoin: o_orderkey = l_orderkey
├── Selection: o_totalprice >= 1000
│   └── Projection: o_orderkey, o_custkey, o_totalprice
│       └── TableScan(orders)
└── Selection: l_extendedprice <= 902
    └── Projection: l_linenumber, l_orderkey, l_extendedprice
        └── TableScan(lineitem)

[6.0, 27939.0, 902.0, 27939.0, 5297.0, 167055.84]
[3.0, 131300.0, 902.0, 131300.0, 1165.0, 100957.91]
2 nuplets produits en 1.9 s



# Exercice 4  : Jointure NestedLoops avec index

L'objectif est de montrer qu'une jointure par boucles imbriquées avec index peut être plus rapide pour certaines requêtes.



## 1) Créer un index

Définir une fonction `create_index(table, attribut)` qui crée un index en mémoire.

Indications :
 * l'index ne contient pas les tuples mais des rowid
 * S'inspirer de l'implementation de  l'operateur TableScan pour lire les tuples et connaitre leur rowid (cf solution vue dans le TP1).

Créer des index pour Orders(o_orderkey) et Lineitems(l_orderkey)



In [58]:
def create_index(table, attribut):
  index = {}
  pages = sorted(glob.glob(os.path.join("data", table, "p*"))) # on peut utiliser sorted,  glob.glob et os.path.join
  attributs_list = pd.read_parquet(pages[0]).columns.tolist()
  indice_attribut = attributs_list.index(attribut)

  for page_number, page in enumerate(pages):
    npa = pd.read_parquet(page).to_numpy()
    for line_number, row in enumerate(npa):
      clé = row[indice_attribut]

      if clé in index:
        index[clé].append((page_number, line_number))
      else:
        index[clé] = [(page_number, line_number)]

  return index

index_orders = create_index("orders", "o_orderkey")
index_lineitem = create_index("lineitem", "l_orderkey")

## 2) Opérateur IndexAccess

Implanter cet opérateur

### class IndexAccess

In [59]:
from collections import OrderedDict

class IndexAccess(Operator):
  def __init__(self, table, index, predicat):
    self.table = table
    self.index = index
    self.predicat = predicat
    # self.cache = OrderedDict()
    # self.cache_size = 100

  def get_page(self, page_path):
    # if page_path in self.cache:
    #     self.cache.move_to_end(page_path)
    #     return self.cache[page_path]

    df = pd.read_parquet(page_path)
    # self.cache[page_path] = df

    # if len(self.cache) > self.cache_size:
    #     self.cache.popitem(last=False)

    return df

  def iterate(self) -> Iterator[np.ndarray]:
    clé = self.predicat

    if clé not in self.index:
      print("Erreur: clé inexistante")
      return

    for item in self.index[clé]:
      page_number = max(1,item[0])
      line_number = item[1]

      page_path = os.path.join("data", self.table, f"p{page_number:03d}.parquet")
      df = self.get_page(page_path)
      row = df.iloc[line_number].to_numpy()
      yield row

## 3) Operateur NestedLoopsJoin



In [61]:
class NestedLoopsJoin(Operator):
  def __init__(self, left, right_table, right_index, join_predicate):
    self.left = left
    self.right_table = right_table
    self.right_index = right_index
    self.join_predicate = join_predicate

    self.right = TableScan(right_table)
    self.right_attribute = self.right.attributes()
    self.left_attribute = left.attributes()

  def iterate(self) -> Iterator[np.ndarray]:
    left_rows = self.left
    indice = self.left.attributes().index(self.join_predicate.split()[0])
    for left_row in left_rows.iterate():
      join_value = left_row[indice]
      right_rows = IndexAccess(self.right_table, self.right_index, join_value)
      for right_row in right_rows.iterate():
        resulting_tuple = np.concatenate((left_row, right_row))
        yield resulting_tuple

  def __str__(self):
    return f"NestedLoopsJoin({self.left}, {self.right_table}, {self.join_predicate})"

  def pretty_print_node(self):
    n = Node(f"NestedLoopsJoin: {self.join_predicate}")
    left = self.left.pretty_print_node()
    left.parent = n
    right = self.right.pretty_print_node()
    right.parent = n
    return n

  def attributes(self) -> list:
    return self.left_attribute + self.right_attribute

# J'ai commenté p1 parce qu'il est vraiment long à s'exécuter. (je suppose que c'est car orders et lineitem ont des cardinalités très différentes)

# p1 = Selection(NestedLoopsJoin(Selection(TableScan("orders"), "o_totalprice >= 1000"), "lineitem", index_lineitem, "o_orderkey = l_orderkey"), "l_extendedprice <= 902")
# pretty_print(p1)
# execute_plan(Limit(p1,10))
# duree_plan(p1)

print("---")

p2 = Selection(NestedLoopsJoin(Selection(TableScan("lineitem"), "l_extendedprice <= 902"), "orders", index_orders, "l_orderkey = o_orderkey"), "o_totalprice >= 1000")
pretty_print(p2)
execute_plan(Limit(p2,10))
duree_plan(p2)

print("-------------------------------------------------------")

query= """
SELECT  l_orderkey, l_linenumber, o_totalprice
FROM Lineitem
JOIN Orders on o_orderkey = l_orderkey
JOIN Supplier on l_suppkey = s_suppkey
WHERE o_totalprice >= 1000
AND l_extendedprice <= 902
"""
print(f"requête: {query}")
run_query(query)

---
Selection: o_totalprice >= 1000
└── NestedLoopsJoin: l_orderkey = o_orderkey
    ├── Selection: l_extendedprice <= 902
    │   └── TableScan(lineitem)
    └── TableScan(orders)

[27939, 2000, 257, 6, 1.0, 902.0, 0.08, 0.05, 'A', 'F', Timestamp('1992-05-10 00:00:00'), Timestamp('1992-05-26 00:00:00'), Timestamp('1992-06-09 00:00:00'), 'COLLECT COD', 'TRUCK', 'f the pinto beans are about the slyl', np.int64(27939), np.int64(5297), 'F', np.float64(167055.84), Timestamp('1992-04-11 00:00:00'), '3-MEDIUM', 'Clerk#000000977', np.int32(0), ' to the platelets. ']
[131300, 2000, 1, 3, 1.0, 902.0, 0.07, 0.08, 'N', 'O', Timestamp('1995-10-20 00:00:00'), Timestamp('1995-11-18 00:00:00'), Timestamp('1995-10-22 00:00:00'), 'NONE', 'FOB', ' instructions haggle car', np.int64(90340), np.int64(6046), 'F', np.float64(185791.98), Timestamp('1992-10-19 00:00:00'), '5-LOW', 'Clerk#000000115', np.int32(0), 'l ideas nag furiously alongside of the']
2 nuplets produits en 1.3 s

---------------------------

# Exercice 5 : Plans non linéaires

Soit la requête de jointure entre les relations Lineitem, Orders, Customer, Supplier

In [64]:
query= """
select  l_orderkey, l_linenumber
from lineitem
JOIN orders on o_orderkey = l_orderkey
JOIN customer on c_custkey = o_custkey
--JOIN Nation on c_nationkey = n_nationkey
join Supplier on l_suppkey = s_suppkey
where s_acctbal <100

"""

t1 = HashJoin(TableScan("lineitem"), Selection(TableScan("supplier"), "s_acctbal < 100"), "l_suppkey = s_suppkey")
pretty_print(t1)
print("---")

t2 = HashJoin(TableScan("orders"), TableScan("customer"), "o_custkey = c_custkey")
pretty_print(t2)
print("---")

t3 = HashJoin(t1, t2, "l_orderkey = o_orderkey")
pretty_print(t3)

HashJoin: l_suppkey = s_suppkey
├── TableScan(lineitem)
└── Selection: s_acctbal < 100
    └── TableScan(supplier)

---
HashJoin: o_custkey = c_custkey
├── TableScan(orders)
└── TableScan(customer)

---
HashJoin: l_orderkey = o_orderkey
├── HashJoin: l_suppkey = s_suppkey
│   ├── TableScan(lineitem)
│   └── Selection: s_acctbal < 100
│       └── TableScan(supplier)
└── HashJoin: o_custkey = c_custkey
    ├── TableScan(orders)
    └── TableScan(customer)



## 1) Plan non linéaire




Construire un plan tel que l'ordre de jointure soit ((Lineitem, Supplier) , (Orders, Customer))

## 2) Ordre optimal
Pour une requête avec 5 relations, proposer un ordre qui minimise le nombre total de nuplets matérialisés pendant le traitement des jointures

In [71]:
query= """
select  l_orderkey, l_linenumber
from lineitem
JOIN orders on o_orderkey = l_orderkey
JOIN customer on c_custkey = o_custkey
JOIN Nation on c_nationkey = n_nationkey
join Supplier on l_suppkey = s_suppkey
where s_acctbal <100

"""
explain_query(query)

a = HashJoin(TableScan("customer"), TableScan("nation"), "c_nationkey = n_nationkey")
b = HashJoin(TableScan("lineitem"), TableScan("supplier"), "l_suppkey = s_suppkey")
c = HashJoin(TableScan("orders"), b, "o_orderkey = l_orderkey")
plan_final = HashJoin(c, a, "o_custkey = c_custkey")
pretty_print(plan_final)
execute_plan(Limit(plan_final,10))
duree_plan(plan_final)

┌───────────────────────────┐
│         PROJECTION        │
│    ────────────────────   │
│         l_orderkey        │
│        l_linenumber       │
│                           │
│        ~55,438 rows       │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         HASH_JOIN         │
│    ────────────────────   │
│      Join Type: INNER     │
│                           │
│        Conditions:        ├────────────────────────────────────────────────────────────────────────┐
│   o_custkey = c_custkey   │                                                                        │
│                           │                                                                        │
│        ~55,438 rows       │                                                                        │
└─────────────┬─────────────┘                                                                        │
┌─────────────┴─────────────┐                                                          ┌───────